# MINI Cells — Experiment 011: Self-Stabilizing Training and Cost

This experiment compares fixed-depth and randomized-depth/stability-regularized MiniCells against a parameter-matched Transformer-S LLM baseline. All models use the same TinyStories stream, tokenizer, token budget and batch schedule. Training cost is measured as synchronized model/optimizer-step wall time on the Kaggle T4; validation, plotting and post-training halting probes are excluded.

Important: Transformer-S is a small parameter-matched baseline. The result must not be generalized to all LLMs or frontier-scale training.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

ROOT = Path('/kaggle/working/mini-cells')
REF = os.environ.get('MINICELLS_REF', 'codex/experiment-011-stabilizing-cost')
os.chdir('/kaggle/working')
if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REF, 'https://github.com/ArcheLabs/mini-cells.git', str(ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm,dev]'], cwd=ROOT, check=True)
os.chdir(ROOT)
print('repo:', ROOT)
print('ref:', REF)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
if not torch.cuda.is_available():
    raise RuntimeError('Experiment 011 requires CUDA')


In [ ]:
subprocess.run([
    sys.executable, '-m', 'pytest',
    'tests/test_language_stabilization.py',
    'tests/test_language_2d.py',
    'tests/test_language_halting.py',
    'tests/test_language_scaling.py',
    '-q'
], cwd=ROOT, check=True)


## Run the five-model cost comparison

With two T4 GPUs, models are run in waves of up to two concurrent workers. Each worker measures only its own synchronized training steps, so queue order does not enter the per-model cost metric.


In [ ]:
subprocess.run([sys.executable, 'scripts/run_language_stabilizing_cost.py'], cwd=ROOT, check=True)


In [ ]:
import json
import pandas as pd
from IPython.display import Image, Markdown, display

OUT = ROOT / 'results' / 'language-stabilizing-cost-v1'
decision = json.loads((OUT / 'decision.json').read_text(encoding='utf-8'))
summary = pd.read_csv(OUT / 'model-summary.csv')
display(Markdown(f"## {decision['status']}: {decision['diagnosis']}"))
display(summary)
for name in [
    'quality-cost-frontier.png',
    'ppl-vs-training-seconds.png',
    'training-cost-per-million.png',
    'peak-vram.png',
    'cost-to-quality.png',
    'adaptive-iterations.png',
    'ppl-vs-training-tokens.png',
]:
    path = OUT / name
    if path.exists():
        display(Image(filename=str(path)))
print(json.dumps(decision, indent=2))


In [ ]:
# Publish immediately after reviewing the decision and cost plots so the Kaggle session is not the only copy.
PUBLISH = False
if PUBLISH:
    subprocess.run([
        sys.executable, 'scripts/publish_experiment_011_results.py', '--push'
    ], cwd=ROOT, check=True)
